# **Часть 1. Знакомство с данными в PostgreSQL**

#### **Задание 1.1: загрузите и изучите данные**

In [1]:
# Импортируем библиотеки
import json
from datetime import timedelta
from pathlib import Path
import pandas as pd
from sqlalchemy import create_engine
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

In [2]:
# Установка константных параметров
RANDOM_STATE = 42

# Определяем параметры подключения к БД из файла
CONFIG_PATH = Path("config/config.json")

In [3]:
# Настройки Pandas для отображения DataFrame
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', None)
pd.set_option('display.float_format', lambda x: f'{x:,.3f}'.replace(',', ' '))
pd.set_option('display.memory_usage', True)

# Настройка стиля графиков
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("viridis")

In [4]:
# Функция для загрузки конфигурации
def load_config(path):
    """Читает параметры подключения из локального файла конфигурации"""
    with open(path, "r", encoding="utf-8") as file:
        return json.load(file)

# Функция для загрузки таблицы из PostgreSQL
def load_table(query, conn):
    """Загружаем таблицу из PostgreSQL в DataFrame"""
    return pd.read_sql_query(query, conn)

In [5]:
# Определяем параметры подключения к БД
db_params = load_config(CONFIG_PATH)["student_db_connection"]
table_dict = {}

connection_string = "postgresql+psycopg2://{}:{}@{}:{}/{}".format(
    db_params["user"],
    db_params["password"],
    db_params["host"],
    db_params["port"],
    db_params["dbname"],
) 
engine = create_engine(connection_string)

In [6]:

# Загружаем все таблицы
for table_name in ["customers", "sessions", "events", "orders"]:
    table_dict[table_name] = load_table(f"SELECT * FROM {table_name}", engine)

customers_df = table_dict["customers"]
sessions_df = table_dict["sessions"]
events_df = table_dict["events"]
orders_df = table_dict["orders"]

In [7]:
# Проверяем загрузку таблиц
for table_name, df in table_dict.items():
    print(f"Таблица {table_name} {df.shape}")

Таблица customers (17846, 7)
Таблица sessions (37699, 6)
Таблица events (239089, 10)
Таблица orders (10652, 10)


In [8]:
# Создаем функцию для отображения первичных данных
def primary_info(df, name):
    """
    Выводит базовую сводную информацию о DataFrame: заголовок с именем таблицы,
    первые 5 строк, сводку типов и пропусков, а также описательную статистику.
    """
    print('='*50)
    print(f'Информация о таблице: {name}')
    print('='*50)
    display(df.head(5))
    df.info()
    display(df.describe(include='all').T)
    print()

In [9]:
# Выводим первичную информацию 
for name, df in table_dict.items():
    primary_info(df, name)

Информация о таблице: customers


,customer_id,name,email,country,age,signup_date,marketing_opt_in
0,1,Jennifer Salinas,nicholas59@example.org,JP,71,2020-09-04,True
1,2,Phillip Ramos,christinarubio@example.com,IN,26,2020-04-05,False
2,3,Dawn Fowler,jessica03@example.org,BR,21,2023-08-31,True
3,4,Mario Butler,paula27@example.org,FR,63,2022-06-30,True
4,5,Amber Brown,kevin85@example.net,BR,19,2022-07-22,True


<class 'pandas.DataFrame'>
RangeIndex: 17846 entries, 0 to 17845
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   customer_id       17846 non-null  int64 
 1   name              17846 non-null  str   
 2   email             17846 non-null  str   
 3   country           17846 non-null  str   
 4   age               17846 non-null  int64 
 5   signup_date       17846 non-null  object
 6   marketing_opt_in  17846 non-null  bool  
dtypes: bool(1), int64(2), object(1), str(3)
memory usage: 854.1+ KB


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
customer_id,17 846.000,NaN,NaN,NaN,10 014.596,5 786.160,1.000,4 993.250,10 030.500,15 019.750,19 999.000
name,17846,16146,Michael Smith,8,NaN,NaN,NaN,NaN,NaN,NaN,NaN
email,17846,17846,nicholas59@example.org,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
country,17846,17,US,3264,NaN,NaN,NaN,NaN,NaN,NaN,NaN
age,17 846.000,NaN,NaN,NaN,46.521,16.796,18.000,32.000,47.000,61.000,75.000
signup_date,17846,2131,2024-07-19,21,NaN,NaN,NaN,NaN,NaN,NaN,NaN
marketing_opt_in,17846,2,True,9927,NaN,NaN,NaN,NaN,NaN,NaN,NaN



Информация о таблице: sessions


...

,count,unique,top,freq,mean,min,25%,50%,75%,max,std
event_id,239 089.000,NaN,NaN,NaN,380 308.825,11.000,190 524.000,379 974.000,569 940.000,760 955.000,219 732.927
session_id,239 089.000,NaN,NaN,NaN,60 126.523,2.000,30 357.000,60 100.000,89 990.000,119 999.000,34 623.163
timestamp,239089,NaN,NaN,NaN,2024-12-03 05:24:27.895503,2024-01-01 01:02:40,2024-06-21 13:21:33,2024-12-03 06:49:17,2025-05-19 04:03:32,2025-11-01 01:50:04,NaN
event_type,239089,4,page_view,169300,NaN,NaN,NaN,NaN,NaN,NaN,NaN
product_id,214 208.000,NaN,NaN,NaN,650.943,1.000,367.000,672.000,953.000,1 197.000,340.451
qty,44 908.000,NaN,NaN,NaN,1.298,1.000,1.000,1.000,1.000,4.000,0.781
cart_size,14 229.000,NaN,NaN,NaN,2.273,1.000,1.000,2.000,3.000,13.000,1.605
payment,10652,4,card,7440,NaN,NaN,NaN,NaN,NaN,NaN,NaN
discount_pct,10 652.000,NaN,NaN,NaN,7.140,0.000,0.000,5.000,15.000,20.000,7.475
amount_usd,10 652.000,NaN,NaN,NaN,133.176,2.800,40.267,86.055,174.677,2 680.640,148.664



Информация о таблице: orders


,order_id,customer_id,order_time,payment_method,discount_pct,subtotal_usd,total_usd,country,device,source
0,1,13917,2025-01-31 23:07:42,card,20.000,107.150,85.720,PL,desktop,organic
1,2,1022,2024-02-19 01:17:50,card,0.000,116.170,116.170,FR,tablet,organic
2,3,6145,2024-12-04 20:24:13,card,0.000,137.350,137.350,US,mobile,organic
3,4,3152,2024-07-17 08:50:47,card,15.000,32.180,27.350,BR,mobile,email
4,11,14044,2025-03-12 02:48:39,card,15.000,55.770,47.400,US,mobile,direct


<class 'pandas.DataFrame'>
RangeIndex: 10652 entries, 0 to 10651
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   order_id        10652 non-null  int64         
 1   customer_id     10652 non-null  int64         
 2   order_time      10652 non-null  datetime64[us]
 3   payment_method  10652 non-null  str           
 4   discount_pct    10652 non-null  float64       
 5   subtotal_usd    10652 non-null  float64       
 6   total_usd       10652 non-null  float64       
 7   country         10652 non-null  str           
 8   device          10652 non-null  str           
 9   source          10652 non-null  str           
dtypes: datetime64[us](1), float64(3), int64(2), str(4)
memory usage: 832.3 KB


,count,unique,top,freq,mean,min,25%,50%,75%,max,std
order_id,10 652.000,NaN,NaN,NaN,16 777.188,1.000,8 435.750,16 748.000,25 116.250,33 580.000,9 688.516
customer_id,10 652.000,NaN,NaN,NaN,10 039.666,1.000,5 040.000,10 063.500,15 054.500,19 999.000,5 799.645
order_time,10652,NaN,NaN,NaN,2024-12-03 23:08:50.272343,2024-01-01 07:08:06,2024-06-22 15:20:19.750000,2024-12-04 20:51:28.500000,2025-05-18 16:56:44.500000,2025-10-31 22:59:41,NaN
payment_method,10652,4,card,7440,NaN,NaN,NaN,NaN,NaN,NaN,NaN
discount_pct,10 652.000,NaN,NaN,NaN,7.140,0.000,0.000,5.000,15.000,20.000,7.475
subtotal_usd,10 652.000,NaN,NaN,NaN,143.359,3.500,43.703,93.295,187.988,2 680.640,158.914
total_usd,10 652.000,NaN,NaN,NaN,133.176,2.800,40.267,86.055,174.677,2 680.640,148.664
country,10652,17,US,1977,NaN,NaN,NaN,NaN,NaN,NaN,NaN
device,10652,3,mobile,5911,NaN,NaN,NaN,NaN,NaN,NaN,NaN
source,10652,6,organic,3583,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [10]:
# Преобразуем признак в формат datetime
events_df["timestamp"] = pd.to_datetime(events_df["timestamp"], utc=True, errors="coerce")
sessions_df["start_time"] = pd.to_datetime(sessions_df["start_time"], utc=True, errors="coerce")
orders_df["order_time"] = pd.to_datetime(orders_df["order_time"], utc=True, errors="coerce")
customers_df["signup_date"] = pd.to_datetime(customers_df["signup_date"], utc=True, errors="coerce")


In [11]:
# Определяем кол-во уникальных значений
for name, df in table_dict.items():
    id_cols = [col for col in df.columns if col.endswith("_id")]
    print('='*50)
    print(f'таблице: {name}')
    print()
    print(f'Число строк в таблице: {df.shape[0]}')
    for col in id_cols:
        print(f'Количество уникальных значений в столбце {col}: {df[col].nunique()}')
    print()

таблице: customers

Число строк в таблице: 17846
Количество уникальных значений в столбце customer_id: 17846

таблице: sessions

Число строк в таблице: 37699
Количество уникальных значений в столбце session_id: 37699
Количество уникальных значений в столбце customer_id: 16894

таблице: events

Число строк в таблице: 239089
Количество уникальных значений в столбце event_id: 239089
Количество уникальных значений в столбце session_id: 37699
Количество уникальных значений в столбце product_id: 1197

таблице: orders

Число строк в таблице: 10652
Количество уникальных значений в столбце order_id: 10652
Количество уникальных значений в столбце customer_id: 8230



In [12]:
# Определяем диапазон дат
for df, col in [(customers_df, "signup_date"), (sessions_df, "start_time"), (events_df, "timestamp"), (orders_df, "order_time")]:
    print(f'Для признака {col}:\n  минимальная дата {df[col].min().date()}\n  максимальная дата {df[col].max().date()}\n')

Для признака signup_date:
  минимальная дата 2020-01-01
  максимальная дата 2025-10-31

Для признака start_time:
  минимальная дата 2024-01-01
  максимальная дата 2025-10-31

Для признака timestamp:
  минимальная дата 2024-01-01
  максимальная дата 2025-11-01

Для признака order_time:
  минимальная дата 2024-01-01
  максимальная дата 2025-10-31



In [13]:
# Находим количество пропусков
for name, df in table_dict.items():
    print('='*50)
    print(f'Число пропусков в таблице {name}:\n{df.isnull().sum()}')

Число пропусков в таблице customers:
customer_id         0
name                0
email               0
country             0
age                 0
signup_date         0
marketing_opt_in    0
dtype: int64
Число пропусков в таблице sessions:
session_id     0
customer_id    0
start_time     0
device         0
source         0
country        0
dtype: int64
Число пропусков в таблице events:
event_id             0
session_id           0
timestamp            0
event_type           0
product_id       24881
qty             194181
cart_size       224860
payment         228437
discount_pct    228437
amount_usd      228437
dtype: int64
Число пропусков в таблице orders:
order_id          0
customer_id       0
order_time        0
payment_method    0
discount_pct      0
subtotal_usd      0
total_usd         0
country           0
device            0
source            0
dtype: int64


In [14]:
# Анализируем воронку покупок
count_purchases = events_df["event_type"].value_counts()

print("page_view → add_to_cart:", round(count_purchases["add_to_cart"] / count_purchases["page_view"], 3))
print("add_to_cart → checkout:", round(count_purchases["checkout"] / count_purchases["add_to_cart"], 3))
print("checkout → purchase:", round(count_purchases["purchase"] / count_purchases["checkout"], 3))

page_view → add_to_cart: 0.265
add_to_cart → checkout: 0.317
checkout → purchase: 0.749


In [15]:
# Определяем взаимосвязи таблиц
print("Все события привязаны к сессиям:", events_df["session_id"].isin(sessions_df["session_id"]).all())
print("Все сессии привязаны к пользователям:", sessions_df["customer_id"].isin(customers_df["customer_id"]).all())
print("Все заказы привязаны к пользователям:", orders_df["customer_id"].isin(customers_df["customer_id"]).all())

Все события привязаны к сессиям: True
Все сессии привязаны к пользователям: True
Все заказы привязаны к пользователям: True


#### **Промежуточный итог**

**Общая информация о таблицах**

| Таблица | Строк | Колонок | Ключ | Уникальных ключей |
|---|---|---|---|---|
| customers | 17 846 | 7 | customer_id | 17 846 |
| sessions | 37 699 | 6 | session_id | 37 699 |
| events | 239 089 | 10 | event_id | 239 089 |
| orders | 10 652 | 10 | order_id | 10 652 |

Дубликатов по ключевым полям нет: число уникальных значений совпадает с числом строк в каждой таблице.

**Диапазон дат**

- `signup_date` (customers): 2020-01-01 … 2025-10-31
- `start_time` (sessions): 2024-01-01 … 2025-10-31
- `timestamp` (events): 2024-01-01 … 2025-11-01
- `order_time` (orders): 2024-01-01 … 2025-10-31

Данные покрывают период с января 2024 по ноябрь 2025, поэтому даты среза `2025-09-01` подходит для расчёта окон 7 и 30 дней.

**Пропуски**

- `customers`, `sessions`, `orders` — пропусков нет.
- `events` — пропуски в `product_id` (24 881), `qty` (194 181), `cart_size` (224 860), `payment`, `discount_pct`, `amount_usd` (по 228 437). Это ожидаемо: эти поля заполняются только для определённых типов событий (например, `qty` — для `add_to_cart`, `amount_usd` — для `purchase`).

**Связи таблиц**

- `events → sessions` по `session_id` — все события привязаны к сессиям (True).
- `sessions → customers` по `customer_id` — все сессии привязаны к пользователям (True).
- `orders → customers` по `customer_id` — все заказы привязаны к пользователям (True).

Целостность связей полная: «сиротских» записей нет. `customers` — корневая таблица, `sessions` — мост между событиями и пользователями.

**Воронка покупки**

- `page_view → add_to_cart`: 0.265
- `add_to_cart → checkout`: 0.317
- `checkout → purchase`: 0.749

Основной отвал происходит на первых двух шагах: до корзины доходит лишь ~26.5% просмотров, до оформления — ~31.7% добавлений. При этом начавшие оформление доводят покупку до конца в 74.9% случаев.

**Наблюдения, важные для расчёта признаков**

1. В `events` нет `customer_id` — для привязки событий к пользователям обязателен джойн с `sessions` по `session_id`.
2. Для товарных признаков (уникальные `product_id`) нужно использовать только строки с заданным `product_id` (24 881 пропуск исключается).
3. Денежные признаки считаются по таблице `orders` (поле `total_usd`), а не по `events`.
4. Активность есть не у всех пользователей: сессии — у 16 894 из 17 846, заказы — у 8 230. Значит, в итоговой таблице будут пользователи без активности, у которых признаки заполняются 0 / 0.0 / -1.
5. Все временные поля приведены к единому типу `datetime64[ns, UTC]` — сравнение с `run_date` будет корректным.


# **Часть 2. Подготовка batch-признаков**


#### **Задание 2.1: подготовьте данные**

In [16]:
# Задаем дату среза
run_date = pd.Timestamp('2025-09-01', tz='UTC')

In [17]:
# Удаляем дубликаты из таблиц поля с индентификаторами
customers_df = customers_df.drop_duplicates(subset=['customer_id'])
events_df = events_df.drop_duplicates(subset=['event_id'])
orders_df = orders_df.drop_duplicates(subset=['order_id'])
sessions_df =sessions_df.drop_duplicates(subset=['session_id'])

In [18]:
# Приводим итогувую стоимость заказа к числовому виду
orders_df["total_usd"] = pd.to_numeric(orders_df["total_usd"], errors='coerce').fillna(0.0)

In [19]:
# Объединяем таблицы events → sessions для получения customer_id
events_df = events_df.merge(
    sessions_df[["session_id", "customer_id"]], 
    how='left', 
    on='session_id'
)

In [20]:
# Убираем строки с пропусками в товарных признаках
events_with_product = events_df[events_df["product_id"].notna()]

In [21]:
# Фильтруем таблица по дате среза
sessions_run_date = sessions_df[sessions_df["start_time"] < run_date]
events_run_date = events_df[events_df["timestamp"] < run_date]
events_with_product = events_with_product[events_with_product["timestamp"] < run_date]
orders_run_date = orders_df[orders_df["order_time"] < run_date]

In [22]:
# Проверяем корректность подготовки данных
# Наличие данных из будущего
print("Наличие данных из будущего")
print("   Макс. start_time < run_date:", sessions_run_date["start_time"].max() < run_date)
print("   Макс. timestamp < run_date:", events_run_date["timestamp"].max() < run_date)
print("   Макс. order_time < run_date:", orders_run_date["order_time"].max() < run_date)

# Наличие дублей
print("\nНаличие дублей")
print("   Дубли session_id:", sessions_run_date["session_id"].duplicated().sum())
print("   Дубли event_id:", events_run_date["event_id"].duplicated().sum())
print("   Дубли order_id:", orders_run_date["order_id"].duplicated().sum())


# Размеры после фильтрации
print("\nРазмеры после фильтрации")
print("   sessions:", sessions_run_date.shape, "\n   events:", events_run_date.shape, "\n   orders:", orders_run_date.shape)

Наличие данных из будущего
   Макс. start_time < run_date: True
   Макс. timestamp < run_date: True
   Макс. order_time < run_date: True

Наличие дублей
   Дубли session_id: 0
   Дубли event_id: 0
   Дубли order_id: 0

Размеры после фильтрации
   sessions: (34206, 6) 
   events: (216603, 11) 
   orders: (9645, 10)


#### **Промежуточный итог**

**Выбранная дата среза:** `run_date = 2025-09-01` (UTC).

**Выполненная предобработка:**

1. Удалены дубликаты по ключевым полям: `customer_id`, `session_id`, `event_id`, `order_id` — дублей не обнаружено.
2. Временные поля `timestamp`, `start_time`, `order_time` приведены к единому типу `datetime64[ns, UTC]` (выполнено в Части 1).
3. Поле `total_usd` приведено к числовому типу; нечисловые значения заменены на `0.0`.
4. Для товарных признаков выделены только строки с заданным `product_id` (отдельный датафрейм `events_with_product`).
5. Выполнена фильтрация по времени: только данные строго до `run_date` (`timestamp < run_date`, `start_time < run_date`, `order_time < run_date`).
6. Выполнен джойн `events → sessions` по `session_id` для получения `customer_id` (в событиях его нет).

**Проверки корректности:**

- Данных из будущего нет: максимальные даты во всех таблицах меньше `run_date` (True).
- Дубликатов по ключевым полям нет (0).
- Размеры после фильтрации: sessions — 34 206, events - 216 603, orders — 9 645 строк.

**Вывод:** данные подготовлены корректно, утечки данных из будущего нет, таблицы готовы к расчёту батч-признаков.


#### **Задание 2.2: рассчитайте признаки**

In [23]:
# Определяем границы среза
window_7 = run_date - timedelta(days=7)
window_30 = run_date - timedelta(days=30)

In [25]:
# Функция для расчёта признаков по событиям
def event_features(df, window_start, suffix):
    """
    Функция для расчёта признаков из событий.
    """
    df = df[df["timestamp"] >= window_start]
    grouped = df.groupby("customer_id")
    
    feats = pd.DataFrame({
        f"page_views_{suffix}": grouped["event_type"].apply(lambda x: (x=="page_view").sum()),
        f"add_to_cart_{suffix}": grouped["event_type"].apply(lambda x: (x=="add_to_cart").sum()),
        f"purchases_{suffix}": grouped["event_type"].apply(lambda x: (x=="purchase").sum())
    })

    feats[f"cart_conversion_{suffix}"] = np.where(
        feats[f"page_views_{suffix}"] > 0,
        feats[f"add_to_cart_{suffix}"] / feats[f"page_views_{suffix}"],
        0.0
    )

    feats[f"purchase_conversion_{suffix}"] = np.where(
        feats[f"add_to_cart_{suffix}"] > 0,
        feats[f"purchases_{suffix}"] / feats[f"add_to_cart_{suffix}"],
        0.0
    )
    return feats

In [26]:
# Рассчитываем признаки для 7-дневного и 30-дневного среза
events_7 = event_features(events_run_date, window_7, "7d")
events_30 = event_features(events_run_date, window_30, "30d")

In [27]:
# Функция для расчёта признаков по товарам
def unique_products(df, window_start, suffix):
    df = df[df["timestamp"] >= window_start]
    return df.groupby("customer_id")["product_id"].nunique().rename(f"unique_products_{suffix}")

In [28]:
# Рассчитываем уникальные товары для 7-дневного и 30-дневного среза
products_7 = unique_products(events_with_product, window_7, "7d")
products_30 = unique_products(events_with_product, window_30, "30d")

In [29]:
# Рассчитываем кол-во сессий для 7-дневного и 30-дневного среза
sessions_7 = (sessions_run_date[sessions_run_date["start_time"] >= window_7]
              .groupby("customer_id")
              .size()
              .rename("sessions_7d")
              )
sessions_30 = (sessions_run_date[sessions_run_date["start_time"] >= window_30]
               .groupby("customer_id")
               .size()
               .rename("sessions_30d")
               )


In [30]:
# рассчитываем cреднюю длину сессии за 30 дней
session_duration = (events_run_date
                    .groupby("session_id")["timestamp"]
                    .max()
                    .rename("last_event_time")
                    .reset_index()
                    )

sessions_with_duration = sessions_run_date.merge(session_duration, on="session_id", how="left")
sessions_with_duration["duration_sec"] = (
    sessions_with_duration["last_event_time"] - sessions_with_duration["start_time"]
).dt.total_seconds().fillna(0)

avg_duration_30 = (sessions_with_duration[sessions_with_duration["start_time"] >= window_30]
                   .groupby("customer_id")["duration_sec"]
                   .mean()
                   .rename("avg_session_duration_30d")
                   )

In [31]:
# Рассчитываем признаки по заказам
orders_30 = orders_run_date[orders_run_date["order_time"] >= window_30]

order_features = orders_30.groupby("customer_id").agg(
    orders_30d=("order_id", "count"),
    total_revenue_30d=("total_usd", "sum"),
    avg_order_value_30d=("total_usd", "mean")
)

last_purchase = orders_run_date.groupby("customer_id")["order_time"].max()
days_since_purchase = (run_date - last_purchase).dt.days.rename("days_since_last_purchase")

In [32]:
# Формируем итоговую таблицу
features = customers_df[["customer_id"]].copy()
features["run_date"] = run_date

for df in [events_7, events_30, products_7, products_30,
           sessions_7, sessions_30, avg_duration_30,
           order_features, days_since_purchase]:
    features = features.merge(df, on="customer_id", how="left")

# Заполнение пропусков
count_cols = [c for c in features.columns if c.endswith(("7d", "30d")) and "conversion" not in c and "duration" not in c]
features[count_cols] = features[count_cols].fillna(0)      
features["days_since_last_purchase"] = features["days_since_last_purchase"].fillna(-1)  
features = features.fillna(0.0)                           

In [33]:
# Проверяем итоговую таблицу
print("Дубли:", features.duplicated(subset=["customer_id", "run_date"]).sum())

expected = ["page_views_7d", "page_views_30d", "add_to_cart_7d", "add_to_cart_30d",
            "cart_conversion_7d", "cart_conversion_30d",
            "purchase_conversion_7d", "purchase_conversion_30d",
            "unique_products_7d", "unique_products_30d",
            "sessions_7d", "sessions_30d", "avg_session_duration_30d",
            "days_since_last_purchase", "orders_30d",
            "total_revenue_30d", "avg_order_value_30d"]
missing = [c for c in expected if c not in features.columns]
conv_cols = [c for c in features.columns if "conversion" in c]

if missing: print("Отсутствуют следующие признаки:", missing) 
else: print("Все признаки в таблице существуют")

print("Макс. timestamp < run_date:", events_run_date["timestamp"].max() < run_date)
print("Счётчики >= 0:", (features[count_cols] >= 0).all().all())
print("Конверсии в [0, 1]:", features[conv_cols].apply(lambda s: s.between(0, 1)).all().all())
print("Размер таблицы:", features.shape)

Дубли: 0
Все признаки в таблице существуют
Макс. timestamp < run_date: True
Счётчики >= 0: True
Конверсии в [0, 1]: True
Размер таблицы: (17846, 21)


#### **Промежуточный итог**

**Дата среза:** `run_date = 2025-09-01` (UTC).

**Подход:** признаки считались по каждой таблице отдельно (events, sessions, orders) с группировкой по `customer_id`, затем готовые агрегаты объединялись через `merge` от таблицы `customers`.

**Рассчитанные признаки:**

- По событиям (`events`): количество `page_view` и `add_to_cart` за 7 и 30 дней; конверсии `add_to_cart / page_view` и `purchase / add_to_cart` за 7 и 30 дней.
- По товарам (`events_with_product`): количество уникальных `product_id` за 7 и 30 дней.
- По сессиям (`sessions`): количество сессий за 7 и 30 дней; средняя длина сессии в секундах за 30 дней.
- По заказам (`orders`): количество заказов, сумма и средний `total_usd` за 30 дней; число дней с последней покупки до `run_date`.

**Правила обработки:**

- Окна агрегации: 7 и 30 дней до `run_date` (строго `< run_date`).
- Деление на ноль при расчёте конверсий → `0.0`.
- Счётчики при отсутствии данных → `0`; вещественные и денежные → `0.0`; дней с последней покупки без заказов → `-1`.
- Товарные признаки считались только по строкам с заданным `product_id`.
- Длина сессии без событий → `0`.
- Денежные признаки считались по таблице `orders` (`total_usd`).

**Проверки корректности:**

- Дубликатов по паре `customer_id` + `run_date` нет (0).
- Все признаков из задания присутствуют в итоговой таблице.
- Данных из будущего относительно `run_date` нет (True).
- Счётчики неотрицательны, конверсии в диапазоне [0, 1].

**Результат:** итоговая таблица `features` размером 17 846 × 21 (одна строка на каждого пользователя и дату среза).
